# QA — Reconciliation Metabase vs Adjust (CDS Android)Notebook orchestration: gọi `clients/` để lấy data, gọi `reconciliation/` để xử lý.Toàn bộ logic thật nằm trong 2 package đó — sửa logic ở đó, không sửa ở đây.Đối chiếu đủ 12 chỉ số: ROAS D0-D28, Revenue D0-D28, Cost, Installs — sai số tương đốithống nhất `ratio = mb/adj`, ngưỡng `config.THRESHOLD_PCT`. Cohort chưa đủ maturity theo`Cohort Age` bị loại khỏi so sánh tự động.

In [ ]:
import syssys.path.append("..")  # để import được config.py, clients/, reconciliation/ ở thư mục gốcimport configfrom clients.adjust_client import AdjustClientfrom clients.metabase_client import MetabaseClientfrom reconciliation.normalize import normalize_metabase, normalize_adjust, filter_excluded_datesfrom reconciliation.merge import merge_sources, report_join_gapsfrom reconciliation.flags import build_detailfrom reconciliation.summary import summarize, top_discrepancy

## 0. (Tuỳ chọn) Override config riêng cho lần chạy này`config.EXCLUDED_DATES` và `config.DATA_ASOF_DATE` là kiến thức riêng theo từng lần chạy(ngày nào bị lỗi, chốt dữ liệu vào lúc nào) — override trực tiếp ở đây thay vì sửa `config.py`nếu chỉ dùng cho 1 lần đối chiếu cụ thể.

In [ ]:
# Ví dụ: loại các ngày biết trước là lỗi/thiếu dữ liệu# config.EXCLUDED_DATES = ["2026-09-08", "2026-09-07", "2026-09-05"]# Ví dụ: chốt lại đúng ngày export dữ liệu để tính Cohort Age nhất quán# config.DATA_ASOF_DATE = "2026-09-06"

## 1. Lấy dữ liệu Metabase

In [ ]:
mb = MetabaseClient(base_url=config.METABASE_URL, api_key=config.METABASE_API_KEY)parameters = mb.build_parameters(    config.METABASE_CARD_ID_CDS_ANDROID,    cohort_date="2026-07-17~2026-08-12",    active_only=[True],    # game=["Game A"],   # bỏ comment nếu cần lọc riêng 1 game)df_mb = mb.query_card(config.METABASE_CARD_ID_CDS_ANDROID, parameters)print("Số dòng Metabase:", df_mb.shape[0])df_mb.head()

## 2. Lấy dữ liệu AdjustDùng `config.ADJUST_METRICS_CDS_ANDROID` — đã gồm đủ Installs, Cost, ROAS D0-D28, Revenue D0-D28.

In [ ]:
adjust = AdjustClient(api_token=config.ADJUST_API_TOKEN)df_adjust, adjust_totals = adjust.fetch_report(    app_token=config.ADJUST_APP_TOKEN_CDS_ANDROID,    dimensions=["day", "network", "campaign_network"],    metrics=config.ADJUST_METRICS_CDS_ANDROID,    date_period="2026-07-17:2026-08-12",    extra_params=config.ADJUST_EXTRA_PARAMS_CDS_ANDROID,)print("Số dòng Adjust:", df_adjust.shape[0])print("Totals:", adjust_totals)df_adjust.head()

## 3. Normalize + loại ngày lỗi + Merge

In [ ]:
mb_norm = normalize_metabase(df_mb)adj_norm = normalize_adjust(df_adjust)mb_norm = filter_excluded_dates(mb_norm, config.EXCLUDED_DATES)adj_norm = filter_excluded_dates(adj_norm, config.EXCLUDED_DATES)merged = merge_sources(mb_norm, adj_norm)join_stats = report_join_gaps(merged)

## 4. Tính bảng đối chiếu chi tiết (long-format)Mỗi dòng = 1 cohort x 1 metric. Dùng `merged` đầy đủ (kể cả phần lệch join key) —các dòng đó tự ra flag "N/A (thiếu dữ liệu)", không bị âm thầm bỏ qua.

In [ ]:
detail = build_detail(merged)detail.head(10)

## 5. Tổng hợp % Discrepancy — theo Campaign x Metric, và toàn hệ thống

In [ ]:
summary_result = summarize(detail)summary_by_campaign = summary_result["by_campaign"]summary_overall = summary_result["overall"]

## 6. Top các dòng lệch nhiều nhất — ưu tiên kiểm tra trước

In [ ]:
top = top_discrepancy(detail, top_n=20, per_campaign=5)print("=== Top 20 lệch nhiều nhất (toàn bộ campaign) ===")display(top["top_overall"])print("\n=== Top 5 lệch nhiều nhất THEO TỪNG CAMPAIGN ===")display(top["top_per_campaign"])

## 7. (Tuỳ chọn) Xuất kết quảThêm cell ghi `detail`/`summary_by_campaign`/`summary_overall` ra BigQuery / Google Sheets /`.docx` report tại đây, khi bạn đã sẵn sàng nối vào pipeline output hiện có.```python# detail.to_csv("../outputs/reconciliation_detail.csv", index=False, encoding="utf-8-sig")# summary_by_campaign.to_csv("../outputs/reconciliation_summary_by_campaign.csv", index=False, encoding="utf-8-sig")# summary_overall.to_csv("../outputs/reconciliation_summary_overall.csv", index=False, encoding="utf-8-sig")```